In [0]:
# Create SUBMSN_Mar5 table
spark.sql("""
CREATE TABLE xliidw_dev_lpl.staging_wins.SUBMSN_Mar5 (
    RECTYPE STRING NOT NULL,
    NAME STRING NOT NULL,
    PRODUCT STRING NOT NULL,
    SYMBOL STRING NOT NULL,
    LAYER STRING NOT NULL,
    LATEST STRING NOT NULL,
    FSTSDATE TIMESTAMP,
    QUOTED STRING NOT NULL,
    FQDATE TIMESTAMP,
    HAULER STRING,
    INSAD2 STRING NOT NULL,
    INSAD3 STRING,
    INSCTY STRING,
    INSST STRING,
    INSZIP STRING,
    COUTRY STRING,
    ACCTNO STRING NOT NULL,
    ACTSALUT STRING,
    ACTFNAME STRING,
    ACTMNAME STRING,
    ACTLNAME STRING,
    ACTSNAME STRING,
    ACTTITLE STRING,
    ACTTEL STRING,
    ACTGRS STRING,
    AGENT STRING NOT NULL,
    AGYNAM STRING,
    AGYAD1 STRING,
    AGYAD2 STRING,
    AGYAD3 STRING,
    AGYCTY STRING,
    AGYST STRING,
    AGYZIP STRING,
    AGTSALU STRING,
    AGTFNAME STRING,
    AGTMNAME STRING,
    AGTLNAME STRING,
    AGTSNAME STRING,
    AGTTITLE STRING,
    AGTTEL STRING,
    CNNAM2 STRING,
    CNTEL3 STRING,
    MRKREG STRING,
    POLICY STRING NOT NULL,
    FEFFDTE TIMESTAMP NOT NULL,
    EFFMO STRING,
    DIVISN STRING NOT NULL,
    DEPART STRING NOT NULL,
    LEADUW STRING NOT NULL,
    UNDWRT STRING NOT NULL,
    ESTPREMIUM FLOAT NOT NULL,
    PARTOF FLOAT,
    NEWRNW STRING NOT NULL,
    SICFULL STRING NOT NULL,
    SICDESC STRING NOT NULL,
    MKTREP STRING,
    SUBCAT STRING,
    COMMENTS1 STRING,
    COMMENTS2 STRING,
    COMMENTS3 STRING,
    FRCVDDATE TIMESTAMP NOT NULL,
    RCVMO STRING,
    IDADD STRING NOT NULL,
    FDTEADD TIMESTAMP,
    IDCHG STRING,
    FDTECHG TIMESTAMP,
    INDUSTRYGROUP STRING,
    LOB STRING
)
""")

# Create SUBMSN_Mar13 table
spark.sql("""
CREATE TABLE xliidw_dev_lpl.staging_wins.SUBMSN_Mar13 (
    RECTYPE STRING NOT NULL,
    NAME STRING NOT NULL,
    PRODUCT STRING NOT NULL,
    SYMBOL STRING NOT NULL,
    LAYER STRING NOT NULL,
    LATEST STRING NOT NULL,
    FSTSDATE TIMESTAMP,
    QUOTED STRING NOT NULL,
    FQDATE TIMESTAMP,
    HAULER STRING,
    INSAD2 STRING NOT NULL,
    INSAD3 STRING,
    INSCTY STRING,
    INSST STRING,
    INSZIP STRING,
    COUTRY STRING,
    ACCTNO STRING NOT NULL,
    ACTSALUT STRING,
    ACTFNAME STRING,
    ACTMNAME STRING,
    ACTLNAME STRING,
    ACTSNAME STRING,
    ACTTITLE STRING,
    ACTTEL STRING,
    ACTGRS STRING,
    AGENT STRING NOT NULL,
    AGYNAM STRING,
    AGYAD1 STRING,
    AGYAD2 STRING,
    AGYAD3 STRING,
    AGYCTY STRING,
    AGYST STRING,
    AGYZIP STRING,
    AGTSALU STRING,
    AGTFNAME STRING,
    AGTMNAME STRING,
    AGTLNAME STRING,
    AGTSNAME STRING,
    AGTTITLE STRING,
    AGTTEL STRING,
    CNNAM2 STRING,
    CNTEL3 STRING,
    MRKREG STRING,
    POLICY STRING NOT NULL,
    FEFFDTE TIMESTAMP NOT NULL,
    EFFMO STRING,
    DIVISN STRING NOT NULL,
    DEPART STRING NOT NULL,
    LEADUW STRING NOT NULL,
    UNDWRT STRING NOT NULL,
    ESTPREMIUM FLOAT NOT NULL,
    PARTOF FLOAT,
    NEWRNW STRING NOT NULL,
    SICFULL STRING NOT NULL,
    SICDESC STRING NOT NULL,
    MKTREP STRING,
    SUBCAT STRING,
    COMMENTS1 STRING,
    COMMENTS2 STRING,
    COMMENTS3 STRING,
    FRCVDDATE TIMESTAMP NOT NULL,
    RCVMO STRING,
    IDADD STRING NOT NULL,
    FDTEADD TIMESTAMP,
    IDCHG STRING,
    FDTECHG TIMESTAMP,
    INDUSTRYGROUP STRING,
    LOB STRING
)
""")

DataFrame[]

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, TimestampType, FloatType

# Read CSV file
df_mar5 = spark.read.option("header", True).csv("/Volumes/xliidw_dev_lpl/staging_wins/files/SUBMSN- March5.csv")

# Get target table schema to preserve data types
target_schema = spark.table("xliidw_dev_lpl.staging_wins.SUBMSN_Mar5").schema

# Cast columns and handle nulls for NOT NULL constraints
for field in target_schema:
    if field.name in df_mar5.columns:
        if field.name == "FEFFDTE":
            # Informatica logic: IIF(LTRIM(RTRIM(FIELD45))='', NULL, TO_DATE(FIELD45,'MM/DD/YYYY'))
            df_mar5 = df_mar5.withColumn(field.name,
                F.when(
                    F.trim(F.col(field.name)) == F.lit(''),
                    F.lit(None).cast(TimestampType())
                ).otherwise(
                    F.to_timestamp(F.col(field.name), 'MM/dd/yyyy')
                )
            )
        else:
            df_mar5 = df_mar5.withColumn(field.name, F.col(field.name).cast(field.dataType))

        # For NOT NULL columns, populate nulls with appropriate defaults
        if not field.nullable:
            if isinstance(field.dataType, StringType):
                df_mar5 = df_mar5.withColumn(field.name,
                    F.when(F.col(field.name).isNull(), F.lit('')).otherwise(F.col(field.name)))
            elif isinstance(field.dataType, FloatType):
                df_mar5 = df_mar5.withColumn(field.name,
                    F.when(F.col(field.name).isNull(), F.lit(0.0).cast(FloatType())).otherwise(F.col(field.name)))
            elif isinstance(field.dataType, TimestampType):
                df_mar5 = df_mar5.withColumn(field.name,
                    F.when(F.col(field.name).isNull(), F.lit('1900-01-01 00:00:00').cast(TimestampType())).otherwise(F.col(field.name)))

# Write without overwriteSchema to preserve target table types
df_mar5.write.format("delta").mode("overwrite") \
    .saveAsTable("xliidw_dev_lpl.staging_wins.SUBMSN_Mar5")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, TimestampType, FloatType

# Read CSV file
df_mar13 = spark.read.option("header", True).csv("/Volumes/xliidw_dev_lpl/staging_wins/files/SUBMSN-March13.csv")

# Get target table schema to preserve data types
target_schema_13 = spark.table("xliidw_dev_lpl.staging_wins.SUBMSN_Mar13").schema

# Cast columns and handle nulls for NOT NULL constraints
for field in target_schema_13:
    if field.name in df_mar13.columns:
        if field.name == "FEFFDTE":
            df_mar13 = df_mar13.withColumn(field.name,
                F.when(
                    F.trim(F.col(field.name)) == F.lit(''),
                    F.lit(None).cast(TimestampType())
                ).otherwise(
                    F.to_timestamp(F.col(field.name), 'MM/dd/yyyy')
                )
            )
        else:
            df_mar13 = df_mar13.withColumn(field.name, F.col(field.name).cast(field.dataType))

        if not field.nullable:
            if isinstance(field.dataType, StringType):
                df_mar13 = df_mar13.withColumn(field.name,
                    F.when(F.col(field.name).isNull(), F.lit('')).otherwise(F.col(field.name)))
            elif isinstance(field.dataType, FloatType):
                df_mar13 = df_mar13.withColumn(field.name,
                    F.when(F.col(field.name).isNull(), F.lit(0.0).cast(FloatType())).otherwise(F.col(field.name)))
            elif isinstance(field.dataType, TimestampType):
                df_mar13 = df_mar13.withColumn(field.name,
                    F.when(F.col(field.name).isNull(), F.lit('1900-01-01 00:00:00').cast(TimestampType())).otherwise(F.col(field.name)))

df_mar13.write.format("delta").mode("overwrite") \
    .saveAsTable("xliidw_dev_lpl.staging_wins.SUBMSN_Mar13")

In [0]:
from pyspark.sql import functions as F# Load tables

df5 = spark.table("xliidw_dev_lpl.staging_wins.SUBMSN_Mar5")
df13 = spark.table("xliidw_dev_lpl.staging_wins.SUBMSN_Mar13")

# Row counts
count5 = df5.count()
count13 = df13.count()

# Column counts
cols5 = len(df5.columns)
cols13 = len(df13.columns)

# Columns present in both, only in Mar5, only in Mar13
cols5_set = set(df5.columns)
cols13_set = set(df13.columns)
common_cols = cols5_set & cols13_set
only5 = cols5_set - cols13_set
only13 = cols13_set - cols5_set

# Count of distinct POLICY in each
distinct_policy5 = df5.select("POLICY").distinct().count()
distinct_policy13 = df13.select("POLICY").distinct().count()

# Count of matching POLICY in both
matching_policy = df5.select("POLICY").intersect(df13.select("POLICY")).count()

# Count of POLICY only in Mar5, only in Mar13
only5_policy = df5.select("POLICY").exceptAll(df13.select("POLICY")).distinct().count()
only13_policy = df13.select("POLICY").exceptAll(df5.select("POLICY")).distinct().count()

# Compare ESTPREMIUM sums
sum_premium5 = df5.agg(F.sum("ESTPREMIUM")).first()[0]
sum_premium13 = df13.agg(F.sum("ESTPREMIUM")).first()[0]

# Compare row-level differences (by POLICY)
diff5 = df5.join(df13, on="POLICY", how="left_anti")
diff13 = df13.join(df5, on="POLICY", how="left_anti")

# Display results
print("Row count Mar5:", count5)
print("Row count Mar13:", count13)
print("Column count Mar5:", cols5)
print("Column count Mar13:", cols13)
print("Common columns:", common_cols)
print("Columns only in Mar5:", only5)
print("Columns only in Mar13:", only13)
print("Distinct POLICY Mar5:", distinct_policy5)
print("Distinct POLICY Mar13:", distinct_policy13)
print("Matching POLICY in both:", matching_policy)
print("POLICY only in Mar5:", only5_policy)
print("POLICY only in Mar13:", only13_policy)
print("Sum ESTPREMIUM Mar5:", sum_premium5)
print("Sum ESTPREMIUM Mar13:", sum_premium13)

print("Rows in Mar5 not in Mar13 (by POLICY):")
display(diff5)
print("Rows in Mar13 not in Mar5 (by POLICY):")
display(diff13)

Row count Mar5: 249335
Row count Mar13: 249335
Column count Mar5: 68
Column count Mar13: 68
Common columns: {'ACTGRS', 'ESTPREMIUM', 'QUOTED', 'INSZIP', 'AGYST', 'LOB', 'FQDATE', 'FDTEADD', 'INSCTY', 'SICFULL', 'CNTEL3', 'ACCTNO', 'DEPART', 'FDTECHG', 'AGTLNAME', 'AGENT', 'AGYNAM', 'INSST', 'ACTMNAME', 'COUTRY', 'INSAD2', 'DIVISN', 'SYMBOL', 'AGYCTY', 'INDUSTRYGROUP', 'NEWRNW', 'RECTYPE', 'SICDESC', 'AGTTEL', 'FSTSDATE', 'AGYZIP', 'COMMENTS1', 'ACTSNAME', 'POLICY', 'IDADD', 'CNNAM2', 'SUBCAT', 'AGYAD2', 'UNDWRT', 'ACTSALUT', 'FEFFDTE', 'LATEST', 'ACTLNAME', 'MKTREP', 'HAULER', 'AGYAD3', 'IDCHG', 'PRODUCT', 'ACTTEL', 'COMMENTS2', 'LEADUW', 'ACTFNAME', 'AGTSNAME', 'LAYER', 'AGTFNAME', 'AGTTITLE', 'MRKREG', 'AGTMNAME', 'AGYAD1', 'EFFMO', 'COMMENTS3', 'RCVMO', 'AGTSALU', 'INSAD3', 'NAME', 'ACTTITLE', 'PARTOF', 'FRCVDDATE'}
Columns only in Mar5: set()
Columns only in Mar13: set()
Distinct POLICY Mar5: 29173
Distinct POLICY Mar13: 29173
Matching POLICY in both: 29173
POLICY only in Mar5: 0
P

POLICY,RECTYPE,NAME,PRODUCT,SYMBOL,LAYER,LATEST,FSTSDATE,QUOTED,FQDATE,HAULER,INSAD2,INSAD3,INSCTY,INSST,INSZIP,COUTRY,ACCTNO,ACTSALUT,ACTFNAME,ACTMNAME,ACTLNAME,ACTSNAME,ACTTITLE,ACTTEL,ACTGRS,AGENT,AGYNAM,AGYAD1,AGYAD2,AGYAD3,AGYCTY,AGYST,AGYZIP,AGTSALU,AGTFNAME,AGTMNAME,AGTLNAME,AGTSNAME,AGTTITLE,AGTTEL,CNNAM2,CNTEL3,MRKREG,FEFFDTE,EFFMO,DIVISN,DEPART,LEADUW,UNDWRT,ESTPREMIUM,PARTOF,NEWRNW,SICFULL,SICDESC,MKTREP,SUBCAT,COMMENTS1,COMMENTS2,COMMENTS3,FRCVDDATE,RCVMO,IDADD,FDTEADD,IDCHG,FDTECHG,INDUSTRYGROUP,LOB


Rows in Mar13 not in Mar5 (by POLICY):


POLICY,RECTYPE,NAME,PRODUCT,SYMBOL,LAYER,LATEST,FSTSDATE,QUOTED,FQDATE,HAULER,INSAD2,INSAD3,INSCTY,INSST,INSZIP,COUTRY,ACCTNO,ACTSALUT,ACTFNAME,ACTMNAME,ACTLNAME,ACTSNAME,ACTTITLE,ACTTEL,ACTGRS,AGENT,AGYNAM,AGYAD1,AGYAD2,AGYAD3,AGYCTY,AGYST,AGYZIP,AGTSALU,AGTFNAME,AGTMNAME,AGTLNAME,AGTSNAME,AGTTITLE,AGTTEL,CNNAM2,CNTEL3,MRKREG,FEFFDTE,EFFMO,DIVISN,DEPART,LEADUW,UNDWRT,ESTPREMIUM,PARTOF,NEWRNW,SICFULL,SICDESC,MKTREP,SUBCAT,COMMENTS1,COMMENTS2,COMMENTS3,FRCVDDATE,RCVMO,IDADD,FDTEADD,IDCHG,FDTECHG,INDUSTRYGROUP,LOB
